# Streaming

Receive responses in real time as they are generated, instead of waiting for the full response. This notebook demonstrates how to enable streaming and read the events the SDK returns.

# Install the TwelveLabs Python SDK

In [ ]:
%pip install twelvelabs

In [ ]:
import os

from twelvelabs import TwelveLabs

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")

# Replace with your knowledge store ID
STORE_ID = "your_knowledge_store_id"

client = TwelveLabs(api_key=API_KEY)

## When You Need Streaming

- Building a chat-like UI where tokens appear as they are generated
- Processing long responses where you want to show progress
- Reducing perceived latency for end users

## Enable Streaming

To enable streaming, call `client.responses.create_stream()` instead of `client.responses.create()`. It accepts the same parameters and returns an iterator of events instead of a single response -- the SDK manages the underlying connection for you.

In [ ]:
stream = client.responses.create_stream(
    knowledge_store_id=STORE_ID,
    input=[
        {
            "type": "message",
            "role": "user",
            "content": "Describe what happens in these videos and images",
        }
    ],
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)

## Reading Stream Events

The stream is a sequence of typed events. Each `response.output_text.delta` event carries a fragment of the generated text in its `delta` field. A final `response.completed` event marks the end of the stream.

In [ ]:
def collect_events(stream) -> list:
    """Collect every event from a Jockey response stream.

    Args:
        stream: The iterator of events returned by client.responses.create_stream().

    Returns:
        A list of the events received, in arrival order.
    """
    events = []
    for event in stream:
        events.append(event)
        print(event)
    return events

In [ ]:
# Make a fresh streaming request and collect the events
stream = client.responses.create_stream(
    knowledge_store_id=STORE_ID,
    input=[
        {
            "type": "message",
            "role": "user",
            "content": "Describe what happens in these videos and images",
        }
    ],
)

events = collect_events(stream)
print(f"\nTotal events received: {len(events)}")

## Streaming with Text Accumulation

In a real application you typically want to accumulate the text content as it streams in. Here is a helper that collects the full text from the response events.

In [ ]:
def stream_text(stream) -> str:
    """Stream events and accumulate the full text response.

    Args:
        stream: The iterator of events returned by client.responses.create_stream().

    Returns:
        The accumulated text content.
    """
    full_text = ""
    for event in stream:
        if event.type == "response.output_text.delta":
            full_text += event.delta
            print(event.delta, end="", flush=True)
    print()  # Final newline
    return full_text

## Common Pitfalls

- **Iterate to completion** -- consume the full stream so the connection closes cleanly and you receive every event
- **Concatenate the deltas** -- each `response.output_text.delta` event carries a fragment of the generated text; join them in arrival order to build the full response
- **Reconnection is not automatic** -- if the stream drops before completion, start a new request

## Next Steps

- [Structured Output](./structured_output.ipynb) -- Force Jockey to return typed JSON matching a schema you define
- [Multi-Turn Sessions](./multi_turn_sessions.ipynb) -- Continue conversations across multiple requests
- [Error Handling](./error_handling.ipynb) -- Retry strategies, polling helpers, and common error patterns
- [API Reference: POST /responses](https://twelvelabs-preview-7f6af7ac-b5df-4358-a7dc-8573e931a808.docs.buildwithfern.com/api-reference/responses/create-response)